
# 📘 08_Dashboard_Rewriter — Update AI/BI Dashboard References

This notebook automatically rewrites the **AI/BI dashboard JSON** to match your current environment configuration.

It replaces hardcoded catalog/schema names with your selected values, so the same dashboard template can be reused across multiple Databricks workspaces or catalogs.

---

### 📂 Folder Structure

📁 notebooks/

    └── 08_Dashboard_Rewriter.ipynb
📁 dashboards/

    └── Data Quality Dashboards Template.lvdash.json   ← input template
    └── Data Quality Dashboards.lvdash.json            ← rewritten output

---

### ⚙️ What It Does

- Reads the dashboard JSON template  
  `/../../dashboards/Data Quality Dashboards Template.lvdash.json`
- Rewrites all SQL references (`FROM catalog.schema.table`) → `{catalog}.{out_schema}.{view_name}`
- Writes a new `.lvdash.json` ready for import into **AI/BI Dashboard**

In [0]:
# Widgets
dbutils.widgets.text("catalog",     "dbdemos_steventan",                 "Catalog")
dbutils.widgets.text("out_schema",  "lakehouse_monitoring_demo_results", "Output Schema (results)")
dbutils.widgets.text("data_schema", "lakehouse_monitoring",              "Data Schema (raw tables)")

catalog     = dbutils.widgets.get("catalog").strip()
out_schema  = dbutils.widgets.get("out_schema").strip()
data_schema = dbutils.widgets.get("data_schema").strip()

# Resolve the notebook's workspace path, then compute sibling 'dashboards' dir
from pyspark.dbutils import DBUtils
import os

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_notebooks_dir = os.path.dirname(_nb_path)

# If this is .../project/notebooks, then dashboards is .../project/dashboards
if _notebooks_dir.endswith("/notebooks"):
    _project_root = _notebooks_dir[:-len("/notebooks")]
else:
    # fallback: go one level up
    _project_root = os.path.dirname(_notebooks_dir)

dash_dir_ws = f"{_project_root}/dashboards"

# Build *workspace* paths (not OS paths) for the two files
template_ws = f"{dash_dir_ws}/Data Quality Dashboards Template.lvdash.json"
output_ws   = f"{dash_dir_ws}/Data Quality Dashboards.lvdash.json"

print("Notebook path:      ", _nb_path)
print("Dashboards folder:  ", dash_dir_ws)
print("Template (workspace path):", template_ws)
print("Output   (workspace path):", output_ws)

In [0]:
# Convert workspace paths to local driver filesystem paths
def ws_to_local(path_ws: str) -> str:
    # Most workspaces mount workspace files at /Workspace/<workspace-path>
    if path_ws.startswith("/Workspace/"):
        return path_ws
    if path_ws.startswith("/"):
        return "/Workspace" + path_ws
    return "/Workspace/" + path_ws

template_local = ws_to_local(template_ws)
output_local   = ws_to_local(output_ws)

print("Template (local):", template_local)
print("Output   (local):", output_local)

In [0]:
# Build fully qualified view names
dq_all_metrics_fqn = f"{catalog}.{out_schema}.dq_all_metrics"
dq_all_details_fqn = f"{catalog}.{out_schema}.dq_all_metric_details"

# Define your replacement map.
# If your template has placeholders, use Option A.
# If your template has literal names, use Option B.

REPLACEMENTS = {
    # --- Option A: placeholders in the template ---
    "__CATALOG__": catalog,
    "__OUT_SCHEMA__": out_schema,
    "__DATA_SCHEMA__": data_schema,
    "__DQ_ALL_METRICS__": dq_all_metrics_fqn,
    "__DQ_ALL_DETAILS__": dq_all_details_fqn,

    # --- Option B: literal names (uncomment if needed) ---
    # "dq_all_metrics": dq_all_metrics_fqn,
    # "dq_all_metric_details": dq_all_details_fqn,
}

In [0]:
# Read the template JSON as plain text, do string replacements, write to output
with open(template_local, "r", encoding="utf-8") as f:
    text = f.read()

count = 0
for src, dst in REPLACEMENTS.items():
    c = text.count(src)
    if c:
        count += c
        text = text.replace(src, dst)

# Ensure output folder exists
import os
os.makedirs(os.path.dirname(output_local), exist_ok=True)

with open(output_local, "w", encoding="utf-8") as f:
    f.write(text)

print(f"Done. Replacements made: {count}")
print(f"Output file written to:  {output_local}")